In [2]:
import sqlite3
from sqlalchemy import create_engine, inspect, func, select, Table, MetaData
import pandas as pd
import time
import jwt
import requests
from typing import Union
import traceback

class DBParser:
    def __init__(self, db_url: str) -> None:
        if 'sqlite' in db_url:
            self.db_type = 'sqlite'
        elif 'mysql' in db_url:
            self.db_type = 'mysql'

        self.engine = create_engine(db_url, echo=False)
        self.conn = self.engine.connect()
        self.db_url = db_url

        self.inspector = inspect(self.engine)
        self.table_names = self.inspector.get_table_names()

        self._table_fields = {}
        self.foreign_keys = []
        self._table_sample = {}

        for table_name in self.table_names:
            print("Table ->", table_name)
            self._table_fields[table_name] = {}

            self.foreign_keys += [
                {
                    'constrained_table': table_name,
                    'constrained_columns': x['constrained_columns'],
                    'referred_table': x['referred_table'],
                    'referred_columns': x['referred_columns'],
                } for x in self.inspector.get_foreign_keys(table_name)
            ]

            table_instance = Table(table_name, MetaData(), autoload_with=self.engine)
            table_columns = self.inspector.get_columns(table_name)
            self._table_fields[table_name] = {x['name']: x for x in table_columns}

            for column_meta in table_columns:
                column_instance = getattr(table_instance.columns, column_meta['name'])

                query = select(func.count(func.distinct(column_instance)))
                distinct_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['distinct'] = distinct_count

                field_type = self._table_fields[table_name][column_meta['name']]['type']
                field_type = str(field_type)
                if 'text' in field_type.lower() or 'char' in field_type.lower():
                    query = (
                        select(column_instance, func.count().label('count'))
                        .group_by(column_instance)
                        .order_by(func.count().desc())
                        .limit(1)
                    )
                    top1_value = self.conn.execute(query).fetchone()[0]
                    self._table_fields[table_name][column_meta['name']]['mode'] = top1_value

                query = select(func.count()).filter(column_instance == None)
                nan_count = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['nan_count'] = nan_count

                query = select(func.max(column_instance))
                max_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['max'] = max_value

                query = select(func.min(column_instance))
                min_value = self.conn.execute(query).fetchone()[0]
                self._table_fields[table_name][column_meta['name']]['min'] = min_value

                query = select(column_instance).limit(10)
                random_value = self.conn.execute(query).all()
                random_value = [x[0] for x in random_value]
                random_value = [str(x) for x in random_value if x is not None]
                random_value = list(set(random_value))
                self._table_fields[table_name][column_meta['name']]['random'] = random_value[:3]

            query = select(table_instance)
            self._table_sample[table_name] = pd.DataFrame([self.conn.execute(query).fetchone()])
            self._table_sample[table_name].columns = [x['name'] for x in table_columns]

    def get_table_fields(self, table_name) -> pd.DataFrame:
        return pd.DataFrame.from_dict(self._table_fields[table_name]).T

    def get_data_relations(self) -> pd.DataFrame:
        return pd.DataFrame(self.foreign_keys)

    def get_table_sample(self, table_name) -> pd.DataFrame:
        return self._table_sample[table_name]

    def check_sql(self, sql) -> Union[bool, str]:
        try:
            self.engine.execute(sql)
            return True, 'ok'
        except:
            err_msg = traceback.format_exc()
            return False, err_msg

    def execute_sql(self, sql) -> bool:
        result = self.engine.execute(sql)
        return list(result)

def generate_token(apikey: str, exp_seconds: int):
    try:
        id, secret = apikey.split(".")
    except Exception as e:
        raise Exception("invalid apikey", e)

    payload = {
        "api_key": id,
        "exp": int(round(time.time() * 1000)) + exp_seconds * 1000,
        "timestamp": int(round(time.time() * 1000)),
    }
    return jwt.encode(
        payload,
        secret,
        algorithm="HS256",
        headers={"alg": "HS256", "sign_type": "SIGN"},
    )

def ask_glm(question, nretry=5):
    if nretry == 0:
        return None

    url = "https://open.bigmodel.cn/api/paas/v4/chat/completions"
    headers = {
        'Content-Type': 'application/json',
        'Authorization': generate_token("83e5bc58555d8bac289e27bac50f8afc.Khk1JjCxb8MJN8Mi", 1000)
    }
    data = {
        "model": "glm-3-turbo",
        "messages": [{"role": "user", "content": question}]
    }
    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        return response.json()
    except:
        return ask_glm(question, nretry - 1)

parser = DBParser('sqlite:///./chinook.db')

In [ ]:
parser.table_names

In [ ]:
parser.get_table_sample('employees')

In [ ]:
parser.get_table_sample('customers')

## NL2SQL Agent 实现

In [9]:
class NL2SQLAgent:
    def __init__(self, db_parser, llm_func):
        self.db_parser = db_parser
        self.llm_func = llm_func
        
        self.nl2sql_prompt = '''你是一个专业的数据库专家，需要将用户的自然语言问题转换为SQL查询语句。

数据库信息：
- 数据库类型: SQLite (chinook.db)
- 所有表名: {table_names}

表结构信息：
{table_schemas}

用户问题: {question}

请直接输出SQL语句，不需要其他解释。SQL语句必须符合SQLite语法。
'''
        
        self.result_prompt = '''你是一个专业的数据库专家，需要将SQL查询结果转换为自然语言回答。

用户问题: {question}

执行的SQL: {sql}

查询结果: {result}

请用自然语言回答用户的问题。
'''
    
    def _build_table_schemas(self):
        schemas = []
        for table_name in self.db_parser.table_names:
            fields = self.db_parser.get_table_fields(table_name)
            field_info = []
            for idx, row in fields.iterrows():
                field_info.append(f"- {idx}: {row['type']}")
            schema_str = f"表名: {table_name}\n" + "\n".join(field_info)
            schemas.append(schema_str)
        return "\n\n".join(schemas)
    
    def ask(self, question: str) -> str:
        table_schemas = self._build_table_schemas()
        
        prompt = self.nl2sql_prompt.format(
            table_names=self.db_parser.table_names,
            table_schemas=table_schemas,
            question=question
        )
        
        print("=== 调用LLM生成SQL ===")
        response = self.llm_func(prompt)
        
        if response is None:
            return "抱歉，LLM调用失败"
        
        sql = response['choices'][0]['message']['content']
        sql = sql.strip('`').strip('\n').replace('sql\n', '').replace('sql', '')
        print(f"生成的SQL: {sql}")
        
        try:
            flag, err = self.db_parser.check_sql(sql)
            if not flag:
                print(f"SQL执行失败: {err}")
                return f"抱歉，SQL执行失败: {err[:200]}"
            
            result = self.db_parser.execute_sql(sql)
            print(f"查询结果: {result}")
            
            result_prompt = self.result_prompt.format(
                question=question,
                sql=sql,
                result=result
            )
            print("=== 调用LLM生成自然语言回答 ===")
            nl_response = self.llm_func(result_prompt)
            if nl_response is None:
                return f"查询结果: {result}"
            
            nl_answer = nl_response['choices'][0]['message']['content']
            return nl_answer
            
        except Exception as e:
            return f"抱歉，执行出错: {str(e)}"

agent = NL2SQLAgent(parser, ask_glm)

## 测试问答

In [ ]:
question1 = "数据库中总共有多少张表"
print(f"提问1: {question1}")
answer1 = agent.ask(question1)
print(f"\n回答1: {answer1}")

In [ ]:
question2 = "员工表中有多少条记录"
print(f"提问2: {question2}")
answer2 = agent.ask(question2)
print(f"\n回答2: {answer2}")

In [ ]:
question3 = "在数据库中所有客户个数和员工个数分别是多少"
print(f"提问3: {question3}")
answer3 = agent.ask(question3)
print(f"\n回答3: {answer3}")

## 直接SQL验证

In [ ]:
conn = sqlite3.connect('chinook.db')
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table';")
print("数据库总表数:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM employees;")
print("员工表记录数:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM customers;")
print("客户表记录数:", cursor.fetchone()[0])